# <EARTHQUAKE NAME> — OSH Facility Search URLs

<ONE-LINE SUMMARY: what happened, where, when>. This notebook pulls USGS ShakeMap MMI (shaking intensity) contours for each event, unions them into shared-intensity polygons across the sequence, and generates OpenSupplyHub facility-search URLs (`?boundary=...`) for each MMI level so we can find contributing facilities inside the shaking footprint.

**Output format:** this notebook (`.ipynb`) is the default output of this pipeline. If you need a different format instead (a script, a CSV of URLs, etc.), ask for it explicitly.

**Spatial terms used below, in plain terms:**
- **MMI** (Modified Mercalli Intensity) — USGS's shaking-strength scale. Roughly: 4 = felt indoors, no damage; 6 = moderate damage; 8+ = severe damage. Each MMI level becomes its own shape and URL.
- **Buffer** — small padding added around the raw shaking-contour edges. Smooths jagged lines and adds a bit of safety margin; doesn't meaningfully change what area is covered.
- **Simplify** — reduces the number of points used to draw a shape's outline. Trades a little shape detail for a shorter, workable URL (OSH URLs have a hard length limit).
- **Dissolve** — when a shaking level breaks into several disconnected patches, stitches them into one connected shape, since the OSH search tool can only take a single connected shape per URL, not several disconnected ones.
- **Coordinate rounding** — trims each point's position to a few decimal places; GPS-level float precision wastes URL length on precision no facility search needs.

OSH URLs have a hard length limit, so polygons are buffered, simplified, and coordinate-rounded before encoding, with an adaptive fallback that simplifies further if a URL still comes out too long.

**⚠️ Before using any generated URL outside internal exploration (sharing externally, publishing anywhere): visually check the "plot the FINAL shapes" cell near the end against the REAL ShakeMap for this event, and confirm it reasonably lines up.** This pipeline is an automated approximation with real correctness edge cases (see the skill's SKILL.md) — it is not a substitute for eyeballing it against the source. The reference ShakeMap link(s) print in the cell below, right after the event data loads.

In [ ]:
import json
import urllib.parse
import warnings
import requests
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
from shapely.ops import unary_union

warnings.filterwarnings("ignore", message="Geometry is in a geographic CRS")
warnings.filterwarnings("ignore", message="invalid value encountered in buffer")
warnings.filterwarnings("ignore", message="invalid value encountered in simplify_preserve_topology")
warnings.filterwarnings("ignore", message="invalid value encountered in unary_union")

In [ ]:
# <FILL IN>: one entry per event in the sequence (mainshock + any aftershocks
# worth including). Find USGS event IDs via:
#   https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=<date>&endtime=<date>&minmagnitude=<mag>&maxlatitude=<n>&minlatitude=<s>&maxlongitude=<e>&minlongitude=<w>
# Before adding a weak/distant aftershock, check its PAGER alert level and max
# MMI (see SKILL.md "Deciding which events to include") — a green-alert
# aftershock that never reaches high MMI mostly just pads out the low-MMI/
# weak-shaking contours without changing anything at the levels that
# correlate with real damage. Ask the user before combining if unsure.
EVENTS = {
    "<M#.# Name>": "<usgs_event_id>",
}

def get_shakemap_contour_url(event_id, product="cont_mmi.json"):
    """Look up the latest shakemap download URL for a given event via the USGS API."""
    api_url = f"https://earthquake.usgs.gov/fdsnws/event/1/query?eventid={event_id}&format=geojson"
    r = requests.get(api_url)
    r.raise_for_status()
    data = r.json()
    shakemap = data["properties"]["products"]["shakemap"][0]
    return shakemap["contents"][f"download/{product}"]["url"]

def get_mmi_contours(event_id):
    url = get_shakemap_contour_url(event_id)
    print(f"Fetching: {url}")
    r = requests.get(url)
    r.raise_for_status()
    return r.json()

# Download all events
contours = {name: get_mmi_contours(eid) for name, eid in EVENTS.items()}
print("Done. MMI levels available:")
for name, fc in contours.items():
    levels = [f["properties"]["value"] for f in fc["features"]]
    print(f"  {name}: {levels}")

print()
print("=" * 80)
print("IMPORTANT: before using any generated URL outside internal exploration,")
print("visually compare the final shapes (see the 'plot the FINAL shapes' cell")
print("below) against the REAL ShakeMap for each event at the link(s) below.")
print("This pipeline is an automated approximation with real correctness edge")
print("cases (see SKILL.md) — confirm it reasonably matches before publishing")
print("anything externally.")
for name, eid in EVENTS.items():
    print(f"  {name}: https://earthquake.usgs.gov/earthquakes/eventpage/{eid}/shakemap/intensity")
print("=" * 80)

In [ ]:
def contour_geom_to_polygon(geometry):
    """
    USGS MMI contours are LineString/MultiLineString rings.
    Each ring encloses the area with intensity >= that MMI value.
    This converts them to Polygon(s) and returns a single unified geometry.
    """
    if geometry["type"] == "LineString":
        coords = geometry["coordinates"]
        if coords[0] != coords[-1]:
            coords = coords + [coords[0]]
        return Polygon(coords)

    elif geometry["type"] == "MultiLineString":
        rings = []
        for ring in geometry["coordinates"]:
            if ring[0] != ring[-1]:
                ring = ring + [ring[0]]
            if len(ring) >= 4:
                p = Polygon(ring)
                if p.is_valid and not p.is_empty:
                    rings.append(p)
        if not rings:
            return None

        # A ring fully nested inside another ring at this same MMI level is a
        # real feature of the data — a small island where shaking dips below
        # this threshold, surrounded by higher intensity (e.g. a bedrock
        # outcrop amid softer, more amplifying soil). Blindly unioning every
        # ring together silently absorbs those islands as solid area, which
        # is wrong: it claims those spots are >= this MMI when the source
        # data says they aren't. Instead, subtract a nested ring from
        # whichever larger ring already contains it (creating a real hole);
        # only union it in as new solid area if it isn't nested in anything.
        rings.sort(key=lambda p: p.area, reverse=True)
        result = None
        for p in rings:
            if result is not None and p.within(result):
                result = result.difference(p)
            else:
                result = p if result is None else unary_union([result, p])
        return result

    return None


# Build {event_name: {mmi_value: polygon}} dict
polygons = {}
for name, fc in contours.items():
    polygons[name] = {}
    for feature in fc["features"]:
        mmi = feature["properties"]["value"]
        poly = contour_geom_to_polygon(feature["geometry"])
        if poly:
            polygons[name][mmi] = poly

print("Polygons built.")

In [ ]:
import math
from shapely.geometry import LineString, MultiPolygon
from shapely.ops import nearest_points

# The OSH renderer doesn't support MultiPolygon geometries. Where a shakemap
# contour comes apart into multiple disjoint pieces, dissolve them into one
# polygon by bridging the largest piece to each smaller piece with a
# minimal-width connector strip (star topology — pieces never connect to
# each other, only to the largest piece).
#
# This runs AFTER buffer/simplify (see to_osh_url below), not on the raw
# contour geometry — simplifying first means the strips attach to smooth,
# already-simplified piece boundaries instead of jagged raw ones, and the
# later buffer/simplify pass doesn't have to contend with the thin strips.
STRIP_WIDTH = 0.0001  # degrees; tune down if connectors are visibly fat on the plot

def dissolve_multipolygon(geom, width=STRIP_WIDTH):
    if geom.geom_type != "MultiPolygon":
        return geom

    pieces = sorted(geom.geoms, key=lambda g: g.area, reverse=True)
    hub = pieces[0]
    parts = [hub]

    for piece in pieces[1:]:
        p1, p2 = nearest_points(hub, piece)
        dx, dy = p2.x - p1.x, p2.y - p1.y
        length = math.hypot(dx, dy)
        if length == 0:
            parts.append(piece)
            continue
        dx, dy = dx / length, dy / length
        # extend slightly into each polygon's interior so the strip overlaps
        # them with real area, guaranteeing the union merges into one Polygon
        p1_ext = (p1.x - dx * width, p1.y - dy * width)
        p2_ext = (p2.x + dx * width, p2.y + dy * width)
        strip = LineString([p1_ext, p2_ext]).buffer(width / 2, cap_style=2)
        parts.append(strip)
        parts.append(piece)

    dissolved = unary_union(parts)
    if dissolved.geom_type != "Polygon":
        print(f"WARNING: dissolve left a {dissolved.geom_type} — widen STRIP_WIDTH")
    return dissolved


def remove_holes(geom):
    """Fill in interior rings (Lesotho-style enclaves) — keep only each
    piece's outer boundary. Some holes are real (a small low-intensity
    island inside a higher-intensity blob, now preserved correctly by
    contour_geom_to_polygon); others are numerical artifacts introduced by
    the connector strips above. Either way, for a facility-search boundary
    it's better to over-include those small patches than to carve fiddly
    cutouts into the shape — it costs a few unnecessary facility matches at
    worst, and the shape reads far cleaner."""
    if geom.geom_type == "Polygon":
        return Polygon(geom.exterior)
    elif geom.geom_type == "MultiPolygon":
        return MultiPolygon([Polygon(p.exterior) for p in geom.geoms])
    return geom

In [ ]:
import heapq

# Two interchangeable simplification algorithms (named after the equivalents in
# ArcGIS Pro's Simplify Polygon tool: POINT_REMOVE and EFFECTIVE_AREA):
#   - "douglas_peucker": shapely's built-in .simplify() (GEOS-backed). Removes
#     vertices whose perpendicular distance from the simplified line is below
#     the tolerance.
#   - "visvalingam_whyatt": removes the vertex whose triangle with its two
#     current neighbors has the smallest area, repeating until the smallest
#     remaining triangle area reaches tolerance^2. Tends to preserve overall
#     shape character better than Douglas-Peucker at aggressive tolerances —
#     it removes genuinely low-impact vertices rather than ones that happen
#     to sit close to a chord, which can otherwise cut across a real feature.
#     Trade-off: it retains more vertices than Douglas-Peucker at the same
#     tolerance value, so the adaptive escalation loop in to_osh_url may need
#     more tries (or a larger escalation factor) to converge under
#     MAX_URL_LENGTH on the most complex, multi-piece, low-MMI levels.

def _vw_simplify_ring(coords, tolerance):
    """Visvalingam-Whyatt simplification of a single closed ring
    (coords[0] == coords[-1])."""
    pts = list(coords[:-1])
    n = len(pts)
    if n <= 3:
        return coords

    min_area = tolerance ** 2
    prev = [(i - 1) % n for i in range(n)]
    nxt = [(i + 1) % n for i in range(n)]
    alive = [True] * n

    def tri_area(i):
        ax, ay = pts[prev[i]]
        bx, by = pts[i]
        cx, cy = pts[nxt[i]]
        return abs((bx - ax) * (cy - ay) - (cx - ax) * (by - ay)) / 2.0

    heap = [(tri_area(i), i) for i in range(n)]
    heapq.heapify(heap)

    n_alive = n
    while heap and n_alive > 3:
        area, i = heapq.heappop(heap)
        if not alive[i]:
            continue
        current = tri_area(i)
        if current != area:
            # neighbor topology changed since this was pushed — re-push with
            # the current area rather than acting on a stale value
            heapq.heappush(heap, (current, i))
            continue
        if area >= min_area:
            break
        alive[i] = False
        n_alive -= 1
        p, nx = prev[i], nxt[i]
        nxt[p] = nx
        prev[nx] = p
        heapq.heappush(heap, (tri_area(p), p))
        heapq.heappush(heap, (tri_area(nx), nx))

    kept = [pts[i] for i in range(n) if alive[i]]
    kept.append(kept[0])
    return kept


def _vw_simplify_polygon(poly, tolerance):
    exterior = _vw_simplify_ring(list(poly.exterior.coords), tolerance)
    interiors = [_vw_simplify_ring(list(r.coords), tolerance) for r in poly.interiors]
    result = Polygon(exterior, interiors)
    if not result.is_valid:
        result = result.buffer(0)
    return result


def _vw_simplify_geom(geom, tolerance):
    if geom.geom_type == "Polygon":
        return _vw_simplify_polygon(geom, tolerance)
    elif geom.geom_type == "MultiPolygon":
        parts = [_vw_simplify_polygon(p, tolerance) for p in geom.geoms]
        return MultiPolygon(parts) if len(parts) > 1 else parts[0]
    return geom


def simplify_geom(geom, tolerance, method="douglas_peucker"):
    if method == "douglas_peucker":
        return geom.simplify(tolerance)
    elif method == "visvalingam_whyatt":
        return _vw_simplify_geom(geom, tolerance)
    else:
        raise ValueError(f"Unknown SIMPLIFY_METHOD: {method!r}")

In [ ]:
# Combined: union of every event at each shared MMI level
all_levels = sorted(set(lvl for p in polygons.values() for lvl in p.keys()))

combined_polygons = {}
for lvl in all_levels:
    parts = [p[lvl] for p in polygons.values() if lvl in p]
    combined_polygons[lvl] = unary_union(parts)

print(f"Combined MMI levels: {all_levels}")

In [ ]:
# Quick plot to sanity-check the RAW contours (pre buffer/simplify/dissolve)
titles = list(polygons.keys()) + ["Combined"]
all_poly_dicts = list(polygons.values()) + [combined_polygons]

fig, axes = plt.subplots(1, len(titles), figsize=(6 * len(titles), 6))
if len(titles) == 1:
    axes = [axes]

for ax, title, pd_ in zip(axes, titles, all_poly_dicts):
    for lvl, poly in sorted(pd_.items()):
        gpd.GeoDataFrame(geometry=[poly], crs="EPSG:4326").plot(ax=ax, alpha=0.3)
    ax.set_title(title)
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()

## Generate OSH boundary-search URLs

OSH facility search URLs have a hard length limit. `to_osh_url` buffers, simplifies (via the selected `SIMPLIFY_METHOD`), dissolves, fills holes, and coordinate-rounds a raw contour geometry before encoding it, and if the resulting URL still exceeds `MAX_URL_LENGTH`, it progressively increases the simplification tolerance until the URL fits (or gives up after a fixed number of tries and prints a warning rather than silently truncating anything).

**If a shape looks off for a given MMI level once you check the final plot below against the real ShakeMap, adjust that level individually via `LEVEL_OVERRIDES` in the next cell — don't just retune `BUFFER`/`SIMPLIFY` globally.** No single buffer or simplify value works for every MMI level, ever: the tight shape near the epicenter and the sprawling, multi-piece shape hundreds of km out need different treatment.

In [ ]:
MAX_URL_LENGTH = 2000  # conservative cross-browser/server-safe URL length limit
COORD_PRECISION = 4  # decimal degrees (~11 m) — far finer than a facility search needs

def process_geom(raw_geom, buffer, simplify, method="douglas_peucker"):
    """Buffer, simplify (via the selected algorithm), dissolve any disjoint
    pieces into one Polygon, then fill in interior holes (simplify first
    keeps the connector strips clean, rather than fighting jagged raw
    contour vertices; holes are removed LAST because dissolve's
    connector-strip unions can themselves introduce new artifact holes)."""
    gdf = gpd.GeoDataFrame(geometry=[raw_geom], crs="EPSG:4326")
    buffered = gdf.buffer(buffer).iloc[0]
    simplified = simplify_geom(buffered, simplify, method)
    dissolved = dissolve_multipolygon(simplified)
    return remove_holes(dissolved)

def round_coordinates(geom_mapping, ndigits=COORD_PRECISION):
    """Round every coordinate in a GeoJSON-style geometry mapping to ndigits
    decimal places. Full float64 precision (~17 significant digits) burns
    URL budget on positional accuracy far beyond what a facility search
    needs — rounding it away roughly halves the encoded length per vertex,
    which buys back far more real shape detail than any amount of extra
    simplification tolerance would cost."""
    def process(obj):
        if isinstance(obj[0], (int, float)):
            return [round(c, ndigits) for c in obj]
        return [process(o) for o in obj]
    geom_mapping = dict(geom_mapping)
    geom_mapping["coordinates"] = process(geom_mapping["coordinates"])
    return geom_mapping

def to_osh_url(raw_geom, buffer=0.1, simplify=0.01, method="douglas_peucker",
               max_len=MAX_URL_LENGTH, max_tries=15):
    """Buffer, simplify, dissolve, and encode a raw contour geometry as an OSH facilities search URL.
    Adaptively increases the simplify tolerance if the encoded URL is too long.
    Returns (url, final_geom) — final_geom is exactly what got encoded, for plotting."""

    def build(simp):
        processed = process_geom(raw_geom, buffer, simp, method)
        coords = json.loads(gpd.GeoSeries([processed]).to_json())["features"][0]["geometry"]
        coords = round_coordinates(coords)
        encoded = urllib.parse.quote(json.dumps(coords, separators=(",", ":")))
        url = f"https://opensupplyhub.org/facilities/?boundary={encoded}&sort_by=contributors_desc"
        return url, processed

    url, geom = build(simplify)
    tries = 0
    while len(url) > max_len and tries < max_tries:
        simplify *= 1.3
        url, geom = build(simplify)
        tries += 1

    if len(url) > max_len:
        print(f"WARNING: could not get URL under {max_len} chars after {max_tries} tries "
              f"(final length {len(url)}, simplify={simplify:.3f})")

    return url, geom

In [ ]:
# Generate all OSH URLs
# SIMPLIFY starts fine-grained (real shape detail on the simpler/smaller levels)
# and to_osh_url's adaptive loop escalates it only as far as needed for the
# large, multi-piece, low-MMI levels that would otherwise blow past MAX_URL_LENGTH.
# Coordinate rounding (COORD_PRECISION, set above) does most of the work of
# keeping URLs short, so simplify rarely needs to escalate far.
# Tweak BUFFER/starting SIMPLIFY here if polygons look off in the plot below.
# SIMPLIFY_METHOD: "douglas_peucker" (shapely built-in) or "visvalingam_whyatt"
# (custom; preserves shape character better at aggressive tolerances, but
# retains more vertices per unit of tolerance — may need more max_tries).
BUFFER = 0.1
SIMPLIFY = 0.01
SIMPLIFY_METHOD = "douglas_peucker"

# If a specific MMI level's shape looks off in the plot below, don't just
# retune BUFFER/SIMPLIFY above — that changes every level at once. No single
# buffer/simplify value looks right across every MMI level, ever: a tight,
# simple high-MMI shape near the epicenter and a sprawling, multi-piece
# low-MMI shape hundreds of km out have very different needs. Override just
# the level that looks wrong here instead, e.g.:
#   LEVEL_OVERRIDES = {4.0: {"buffer": 0.2}, 6.5: {"simplify": 0.02}}
LEVEL_OVERRIDES = {}

urls = {}       # (name, lvl) -> url
final_geoms = {}  # (name, lvl) -> geometry actually encoded (for plotting)

print("=" * 80)
for name, pd_ in list(polygons.items()) + [("Combined", combined_polygons)]:
    print(f"\n--- {name} ---")
    for lvl in sorted(pd_.keys(), reverse=True):
        kwargs = {"buffer": BUFFER, "simplify": SIMPLIFY, "method": SIMPLIFY_METHOD}
        kwargs.update(LEVEL_OVERRIDES.get(lvl, {}))
        url, geom = to_osh_url(pd_[lvl], **kwargs)
        urls[(name, lvl)] = url
        final_geoms[(name, lvl)] = geom
        print(f"  MMI {lvl:4.1f} (len {len(url):4d}): {url}")

In [ ]:
# Plot the FINAL shapes (post buffer/simplify/dissolve/remove_holes) — exactly what each URL encodes
titles = list(polygons.keys()) + ["Combined"]

fig, axes = plt.subplots(1, len(titles), figsize=(6 * len(titles), 6))
if len(titles) == 1:
    axes = [axes]

for ax, name in zip(axes, titles):
    for lvl in sorted(k[1] for k in final_geoms if k[0] == name):
        gpd.GeoDataFrame(geometry=[final_geoms[(name, lvl)]], crs="EPSG:4326").plot(ax=ax, alpha=0.3)
    ax.set_title(name)
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()